# Trivariate Gaussian process emulation of the three components of the knee joint contact force

The notebook `jcf-gp-trivariate-colab.ipynb` models the hip, knee and ankle joint contact forces jointly. This notebook keeps one joint, the knee, and models the three components of its contact force jointly: the anterior-posterior, the axial and the medio-lateral component in the tibia frame of the OpenSim model. One trivariate model is fitted at each of the 101 gait points, so the output of each component is its value at that gait point, and the three outputs of a stride at a gait point form one force vector. The fitted models predict the three component curves of a new stride. Each component keeps its own variance, input range and nugget effect, and every pair of components has a cross-correlation, because the three components are produced by the same ground reaction force and the same muscle forces.

With component-specific ranges the covariance matrix has no Kronecker structure, and it is treated as a dense matrix with $N = 3 \times 480 = 1{,}440$ rows at each gait point. Maximum likelihood estimation requires the Cholesky factorization of this matrix at every evaluation of the likelihood, and the 101 fits are warm-started one from the previous. The notebook fits the model with the adaptive block mixed-precision Cholesky factorization of Salvaña (2026), which stores each block of the Cholesky factor in the lowest precision that a guaranteed error budget allows.

The notebook has three parts. It loads the data, it fits the trivariate model at every gait point by maximum likelihood with the mixed-precision factorization, and it predicts the three component curves of the strides of the three held-out participants and checks that the OpenSim curves fall inside the 95 % prediction intervals. It runs on any GPU runtime. In Colab, choose Runtime, then Change runtime type, then a GPU.

## Setup

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize

assert torch.cuda.is_available(), "No GPU is attached. Choose Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{GPU_NAME}, {GPU_GB:.1f} GB")

## Data

The data are those of the first notebook. One row is one stride of the right leg, and there are 480 training strides from twelve participants and 140 test strides from three other participants. The inputs are three ground reaction force curves, five joint angle curves, the belt speed, the body mass and the height. The outputs used here are the three components of the right knee joint contact force computed by OpenSim, in body weights (BW), time normalized to 101 points of the gait cycle. The components are the force of the femur on the tibia in the tibia frame: $f_x$ is anterior-posterior (shear), $f_y$ is along the long axis of the tibia and negative in compression, and $f_z$ is medio-lateral. Their magnitude is the resultant knee contact force of the earlier notebooks.

The variable `JOINT` below selects the knee; the hip and the ankle have the same columns.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_test = pd.read_csv("data/x-test.csv")
y_test = pd.read_csv("data/y-test.csv")
x_train.shape, y_train.shape, x_test.shape, y_test.shape

## Visualize the inputs

A curve called `name` occupies the 101 columns `name_t000` to `name_t100`; `curve` returns it as a `(strides, 101)` array. Every stride of each input curve is shown, colored by belt speed, with the mean curve of each speed on top. Stance is roughly the first 62 % of the cycle (dotted line), swing the rest.

In [ ]:
GRID = np.arange(101)
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]
OUTPUT_CURVES = ["knee_jcf_fx_BW", "knee_jcf_fy_BW", "knee_jcf_fz_BW", "knee_jcf_BW"]
SPEED_COLOURS = {1.0: "#0072B2", 1.25: "#D55E00", 1.4: "#000000", 1.5: "#009E73"}

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

def overlay(ax, df, name):
    C = curve(df, name)
    speeds = df["belt_speed_mps"].to_numpy()
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].T, color=SPEED_COLOURS[v], lw=0.5, alpha=0.25)
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].mean(axis=0), color=SPEED_COLOURS[v], lw=2.5, label=f"{v:g} m/s (n={rows.sum()})")
    ax.axvline(62, color="grey", lw=0.8, ls=":")
    ax.set_title(name, loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)

### Training inputs

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat, INPUT_CURVES):
    overlay(ax, x_train, name)
axes[0][0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Gait cycle [%]")
fig.tight_layout()

### Testing inputs

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat, INPUT_CURVES):
    overlay(ax, x_test, name)
axes[0][0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Gait cycle [%]")
fig.tight_layout()

## Visualize the outputs

The three components of the knee joint contact force are the force of the femur on the tibia in the tibia frame of the OpenSim model, in body weights. $f_x$ is the anterior-posterior component, the shear across the joint; $f_y$ runs along the long axis of the tibia and is negative in compression, so it carries almost all of the load; $f_z$ is the medio-lateral component and is small. The fourth panel is their magnitude, the resultant knee contact force of the earlier notebooks, whose peak in each stride sets the gait point at which the three components are read.

### Training outputs

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 4.5))
for ax, name in zip(axes, OUTPUT_CURVES):
    overlay(ax, y_train, name)
    ax.set_xlabel("Gait cycle [%]")
axes[0].set_ylabel("JCF [BW]")
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

### Testing outputs

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 4.5))
for ax, name in zip(axes, OUTPUT_CURVES):
    overlay(ax, y_test, name)
    ax.set_xlabel("Gait cycle [%]")
axes[0].set_ylabel("JCF [BW]")
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

## Inputs as a feature vector

Each stride becomes the vector $\mathbf{x}_i$ of its eight input curves and three scalars, standardized on the training strides exactly as in the first notebook. The test strides are standardized with the training means and standard deviations.

In [ ]:
JOINT = "knee"                                             # "hip", "knee" or "ankle"
COMPONENTS = ["fx", "fy", "fz"]

def build_features(x, x_ref):
    cols = []
    for name in INPUT_CURVES:
        C, C_ref = curve(x, name), curve(x_ref, name)
        cols.extend(((C - C_ref.mean(axis=0)) / C_ref.std()).T)
    for name in ["belt_speed_mps", "mass_kg", "height_m"]:
        cols.append((x[name].to_numpy() - x_ref[name].mean()) / x_ref[name].std())
    return np.array(cols).T

F_train = build_features(x_train, x_train)
F_test = build_features(x_test, x_train)
F_train.shape, F_test.shape

The Gaussian process at gait point $t$ models the residual of each component, the output minus the mean $m_j(t)$ of the training strides at that gait point. The residuals of component $j$ are divided by the standard deviation $s_j(t)$ of its training outputs at $t$.

In [ ]:
Y_train = np.stack([curve(y_train, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])          # (3 components, 480 strides, 101 gait points), in BW
Y_test = np.stack([curve(y_test, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])
mean_t = Y_train.mean(axis=1)                                                               # mean curve of each component
scale_t = Y_train.std(axis=1)                                                               # s_j(t), in BW
Z_train = (Y_train - mean_t[:, None, :]) / scale_t[:, None, :]                              # standardized residuals
Z_train.shape

## Trivariate Matérn model

Let $i = 1, \dots, n$ index the training strides, with $n = 480$, and let $j \in \{1, 2, 3\}$ index the anterior-posterior, the axial and the medio-lateral component of the knee contact force. Fix a gait point $t$. The input of stride $i$ is the vector $\mathbf{x}_i \in \mathbb{R}^{811}$ of its whole input curves, and $y_{ij}$ is the output of component $j$ at stride $i$ and gait point $t$. Each component follows the decomposition
$$
y_{ij} = m_j + f_j(\mathbf{x}_i) + \varepsilon_{ij}, \qquad \varepsilon_{ij} \sim \mathcal{N}(0, \sigma_{\varepsilon, j}^2),
$$
where $m_j$ is the mean of component $j$ at $t$ and $\sigma_{\varepsilon, j}^2$ is the nugget effect of component $j$, and $(f_1, f_2, f_3)$ is a trivariate Gaussian process with a covariance function for every pair of components.

Write $h$ for the root mean squared difference between the standardized inputs of two strides. The covariance function is
$$
\operatorname{Cov}\!\big(f_j(\mathbf{x}), f_k(\mathbf{x}')\big) = \beta_{jk}\, \sigma_j \sigma_k \;\, \gamma_{jk}\, \mathcal{M}\!\Big(\frac{h}{\ell_{jk}}\Big), \qquad \mathcal{M}(d) = \Big(1 + \sqrt{5}\, d + \tfrac{5}{3} d^2\Big) e^{-\sqrt{5}\, d},
$$
where $\mathcal{M}$ is the Matérn correlation function with smoothness $5/2$. Component $j$ has its own variance $\sigma_j^2$ and input range $\ell_j$, and the parameters $\beta_{jk}$ form a correlation matrix. For a pair of components the range and the factor $\gamma$ follow from the ranges of the two components,
$$
\frac{1}{\ell_{jk}^2} = \frac{1}{2} \Big( \frac{1}{\ell_j^2} + \frac{1}{\ell_k^2} \Big), \qquad \gamma_{jk} = \Big( \frac{2\, \ell_j \ell_k}{\ell_j^2 + \ell_k^2} \Big)^{5/2}.
$$
This construction is valid for every correlation matrix $(\beta_{jk})$ and in any input dimension (Gneiting, Kleiber and Schlather, 2010, and Apanasovich, Genton and Sun, 2012). For a single component the covariance function reduces to $\sigma_j^2\, \mathcal{M}(h / \ell_j)$, the Matérn covariance function of the univariate emulator. The cross-correlation between components $j$ and $k$ at the same stride is $\rho_{jk} = \beta_{jk}\, \gamma_{jk}$.

The model has eleven parameters at each gait point, and they are estimated by maximizing the likelihood of the stacked residuals $\mathbf{y}$, a vector of length $N = 1{,}440$ with covariance matrix $\boldsymbol{\Sigma}$. The variance of the anterior-posterior component is removed analytically. Write $\boldsymbol{\Sigma} = \sigma_1^2\, \bar{\boldsymbol{\Sigma}}$, where $\bar{\boldsymbol{\Sigma}}$ depends on the ranges, the parameters $\beta_{jk}$, the variance ratios $\sigma_j^2 / \sigma_1^2$ and the relative nugget effects $\eta_j = \sigma_{\varepsilon, j}^2 / \sigma_1^2$. For fixed values of these ten parameters the likelihood is maximized by $\hat{\sigma}_1^2 = \mathbf{y}^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y} / N$, and the negative log likelihood is
$$
-\log p = \tfrac{1}{2} \Big( N \log \hat{\sigma}_1^2 + \log \det \bar{\boldsymbol{\Sigma}} + N + N \log 2\pi \Big).
$$
Each evaluation needs the log determinant of $\bar{\boldsymbol{\Sigma}}$ and one linear solve, and both come from its Cholesky factor.

The class below holds the model. It orders the points by component, then stride, and it returns any block of $\bar{\boldsymbol{\Sigma}}$ on the GPU in double precision, so the full matrix is never stored. The covariance matrix depends on the inputs and the parameters only, so the same object serves every gait point; only the output vector changes.

In [ ]:
def matern(d):
    s = 5.0**0.5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)

def correlation_matrix(raw):
    # 3 x 3 correlation matrix from three unconstrained numbers, through the angles of its Cholesky factor
    a = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.array([[1.0, 0.0, 0.0],
                  [np.sin(a[0]), np.cos(a[0]), 0.0],
                  [np.sin(a[1]), np.cos(a[1]) * np.sin(a[2]), np.cos(a[1]) * np.cos(a[2])]])
    return L @ L.T

def pair_ranges(l):
    # 1 / l_jk and gamma_jk for every pair of components
    a2 = 1.0 / l**2
    inverse = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    return inverse, (np.sqrt(a2[:, None] * a2[None, :]) / inverse**2) ** 2.5

class TrivariateMatern:
    def __init__(self, F, n_outputs, block_size=120):
        self.n, self.N = F.shape[0], n_outputs * F.shape[0]
        Ft = torch.tensor(F, dtype=torch.float64, device=DEVICE)
        self.H = torch.cdist(Ft, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F.shape[1]**0.5      # h between strides
        index = torch.arange(self.N, device=DEVICE)
        self.component, self.stride = index // self.n, index % self.n
        self.blocks = [slice(a, min(a + block_size, self.N)) for a in range(0, self.N, block_size)]

    def set_outputs(self, Z):
        self.y = torch.tensor(Z.reshape(-1), dtype=torch.float64, device=DEVICE)                          # stacked standardized residuals

    def set_parameters(self, theta):
        # theta = log input ranges (3), correlation angles (3), log variance ratios of fy and fz (2), log relative nugget effects (3)
        l = np.exp(theta[0:3])
        beta = correlation_matrix(theta[3:6])
        v = np.concatenate([[1.0], np.exp(theta[6:8])])
        eta = np.exp(theta[8:11])
        inverse_l, gamma = pair_ranges(l)
        rho = beta * gamma                                                         # cross-correlations between the components at the same stride
        as_tensor = lambda a: torch.tensor(a, dtype=torch.float64, device=DEVICE)
        self.amplitude = as_tensor(rho * np.sqrt(np.outer(v, v)))
        self.inverse_l = as_tensor(inverse_l)
        self.nugget = as_tensor(eta)[self.component]                                 # relative nugget effect of every point
        self.m = float(eta.min())                                                  # lower bound on the smallest eigenvalue
        self.parameters = dict(l=l, v=v, eta=eta, rho=rho)

    def block(self, rows, cols):
        # block of the covariance matrix between two sets of points, without the nugget effect
        pair = (self.component[rows][:, None], self.component[cols][None, :])
        h = self.H[self.stride[rows]][:, self.stride[cols]]
        return self.amplitude[pair] * matern(h * self.inverse_l[pair])

## Adaptive block mixed-precision Cholesky factorization

A matrix entry takes 8 bytes in double precision (`fp64`), 4 bytes in single precision (`fp32`), 2 bytes in `bfloat16` (`bf16`) and 1 byte in the 8-bit format `fp8`. The factorization follows Algorithm 1 of Salvaña (2026). It never stores the covariance matrix, and it stores each block of the Cholesky factor in the lowest of these precisions that an error budget allows.

Partition $\bar{\boldsymbol{\Sigma}}$ of a gait point into an $n_B \times n_B$ grid of blocks of $n_b = 120$ consecutive points, so that $n_B = 12$. Let $m$ be a known lower bound on the smallest eigenvalue, $0 < m \le \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$. Here $\bar{\boldsymbol{\Sigma}}$ is a nonnegative definite matrix plus a diagonal matrix that holds the relative nugget effects, so $m = \min_j \eta_j$ is available at no cost. For an error-budget parameter $0 < \tau < 1$ the budget of each block is
$$
b = \frac{\tau\, m}{n_B}.
$$

Let $\widetilde{\mathbf{L}}$ denote the stored factor. At block column $k$ the diagonal block is computed and kept in `fp64`,
$$
\widetilde{\mathbf{L}}_{kk} = \operatorname{chol}\Big( \bar{\boldsymbol{\Sigma}}_{kk} - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \widetilde{\mathbf{L}}_{kj}^\top \Big),
$$
and for each $i > k$ the off-diagonal block is computed in `fp64` as
$$
\mathbf{A}_{ik} = \Big( \bar{\boldsymbol{\Sigma}}_{ik} - \sum_{j < k} \widetilde{\mathbf{L}}_{ij} \widetilde{\mathbf{L}}_{kj}^\top \Big) \widetilde{\mathbf{L}}_{kk}^{-\top}.
$$
The block $\mathbf{A}_{ik}$ is then converted to each candidate format $p$ in the order `fp8`, `bf16`, `fp32`, read back in `fp64`, and stored in the first format with finite entries that satisfies
$$
\big\| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big\|_F\, \big\| \widetilde{\mathbf{L}}_{kk} \big\|_2 \le b,
$$
where $\mathrm{fl}_p$ denotes conversion to format $p$, $\|\cdot\|_F$ is the Frobenius norm and $\|\cdot\|_2$ the spectral norm. If no candidate qualifies, the block stays in `fp64`. The left-hand side bounds the change that storing the block in format $p$ causes in the covariance matrix implied by the stored factor, $\widetilde{\boldsymbol{\Sigma}} = \widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top$.

Under this rule, and treating `fp64` arithmetic as exact, the Cholesky factorization of every diagonal block exists, so the factorization cannot break down, and the completed factor satisfies
$$
\big\| \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}} \big\|_F \le \tau\, m \le \tau\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}}),
$$
which implies that $\widetilde{\boldsymbol{\Sigma}}$ is positive definite. All arithmetic is done in `fp64`. Covariance blocks are formed when they are needed and discarded after use, and the log determinant is accumulated from the diagonal blocks, $\log \det \widetilde{\boldsymbol{\Sigma}} = 2 \sum_k \sum_r \log (\widetilde{\mathbf{L}}_{kk})_{rr}$. The notebook uses $\tau = 0.99$.

In [ ]:
TAU = 0.99
CANDIDATES = {"fp8": torch.float8_e4m3fn, "bf16": torch.bfloat16, "fp32": torch.float32}      # lowest precision first
LARGEST = {"fp8": 448.0, "bf16": 3.39e38, "fp32": 3.40e38}                                    # largest finite number of each format
FORMAT_NAMES = ["fp8", "bf16", "fp32", "fp64"]

def choose_format(A, budget):
    # the lowest precision whose measured storage error fits the budget, fp64 if none does
    largest = float(A.abs().max())
    for name, dtype in CANDIDATES.items():
        if largest <= LARGEST[name] and float(torch.linalg.norm(A.to(dtype).to(torch.float64) - A)) <= budget:
            return name, dtype
    return "fp64", torch.float64

def mixed_precision_cholesky(model):
    blocks, nB = model.blocks, len(model.blocks)
    b = TAU * model.m / nB                                                         # budget of each block
    L, logdet = {}, 0.0
    count = {"fp8": 0, "bf16": 0, "fp32": 0, "fp64": 0}
    precision = np.full((nB, nB), np.nan)                                          # format of every factor block, for the figure
    for k in range(nB):
        # diagonal block, kept in fp64
        S = model.block(blocks[k], blocks[k])
        S.diagonal().add_(model.nugget[blocks[k]])
        row = [L[k, j].to(torch.float64) for j in range(k)]
        for Lkj in row:
            S.addmm_(Lkj, Lkj.T, alpha=-1.0)
        Lkk = torch.linalg.cholesky(S)
        L[k, k] = Lkk
        precision[k, k] = FORMAT_NAMES.index("fp64")
        logdet += 2.0 * float(torch.log(torch.diagonal(Lkk)).sum())
        norm_kk = float(torch.linalg.matrix_norm(Lkk, ord=2))
        # off-diagonal blocks of column k, each stored in the lowest precision that fits the budget
        for i in range(k + 1, nB):
            A = model.block(blocks[i], blocks[k])
            for j in range(k):
                A.addmm_(L[i, j].to(torch.float64), row[j].T, alpha=-1.0)
            A = torch.linalg.solve_triangular(Lkk.T, A, upper=True, left=False)
            name, dtype = choose_format(A, b / norm_kk)
            L[i, k] = A.to(dtype)
            count[name] += 1
            precision[i, k] = FORMAT_NAMES.index(name)
    return L, logdet, count, precision

def solve(L, blocks, rhs):
    # solve (L L^T) x = rhs by block forward and back substitution; rhs is a vector or a matrix with one right-hand side per column
    x, nB = rhs.clone().reshape(rhs.shape[0], -1), len(blocks)
    for k in range(nB):
        s = x[blocks[k]]
        for j in range(k):
            s -= L[k, j].to(torch.float64) @ x[blocks[j]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k], s, upper=False)
    for k in range(nB - 1, -1, -1):
        s = x[blocks[k]]
        for i in range(k + 1, nB):
            s -= L[i, k].to(torch.float64).T @ x[blocks[i]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k].T, s, upper=True)
    return x.reshape(rhs.shape)

def negative_log_likelihood(theta, model):
    model.set_parameters(theta)
    L, logdet, count, precision = mixed_precision_cholesky(model)
    alpha = solve(L, model.blocks, model.y)
    sigma2 = float(model.y @ alpha) / model.N                                      # profiled variance of the hip
    stored_gb = sum(block.nbytes for block in L.values()) / 1e9
    N = model.N
    return 0.5 * (N * np.log(sigma2) + logdet + N + N * np.log(2 * np.pi)), sigma2, count, stored_gb, precision

### Size of the covariance matrix

The cell below builds the model on all training strides and prints the size of the covariance matrix that every evaluation of the likelihood has to factor, at each gait point.

In [ ]:
model = TrivariateMatern(F_train, len(COMPONENTS))
N, nB = model.N, len(model.blocks)
print(f"covariance matrix          {N:,} x {N:,} per gait point   (1 joint x {len(COMPONENTS)} components x {model.n} strides)")
print(f"number of entries          {N * N:,}")
print(f"matrix in fp64             {N * N * 8 / 1e6:.1f} MB")
print(f"Cholesky factor in fp64    {N * (N + 1) / 2 * 8 / 1e6:.1f} MB")
print(f"Cholesky factor in fp32    {N * (N + 1) / 2 * 4 / 1e6:.1f} MB")
print(f"blocks                     {nB} x {nB}, each with up to {model.blocks[0].stop:,} points")
print(f"GPU                        {GPU_NAME}, {GPU_GB:.1f} GB")

## Maximum likelihood estimation at every gait point

At each gait point the ten parameters are estimated with the L-BFGS-B optimizer, with gradients from forward differences of the likelihood. The first gait point starts from the input range of the univariate knee emulator of the first notebook for all three components, unit variances of the standardized outputs, nugget effects of one percent and moderate cross-correlations between the components. Every later gait point starts from the estimates of the previous one, which is why most fits converge in a few iterations. When the line search fails at the warm start, the gait point is refitted from the starting values of the first gait point and the better fit is kept. The parameters are bounded to a plausible region, input ranges between 0.05 and 5 standardized units and relative nugget effects between $10^{-5}$ and $0.5$.

Every evaluation of the likelihood is one mixed-precision factorization of the $1{,}440 \times 1{,}440$ matrix. One line is printed per gait point: the negative log likelihood, the number of iterations and evaluations, the formats of the off-diagonal factor blocks and the estimates, with the ranges, variances and nugget effects listed for $f_x$, $f_y$ and $f_z$ in that order and each cross-correlation labeled with its pair of components. The estimates of every finished gait point are saved to a checkpoint file, and the cell resumes from that file when it exists.

In [ ]:
# starting values at the first gait point (fx, fy, fz): the univariate input range of the joint, unit variances of the standardized outputs,
# 1 % nugget effects; every later gait point starts from the estimates of the previous one
l0 = np.full(3, {"hip": 0.354, "knee": 0.287, "ankle": 0.313}[JOINT])   # input ranges [standardized units]
var0 = np.array([1.0, 1.0, 1.0])                           # variances, standardized
nugget0 = np.array([0.01, 0.01, 0.01])                     # nugget effects, standardized
theta_first = np.concatenate([np.log(l0), [0.3, 0.3, 0.3], np.log(var0[1:] / var0[0]), np.log(nugget0 / var0[0])])
bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.5))] * 3

GAIT_POINTS = GRID                                          # all 101 gait points; GRID[::5] fits every fifth
MAX_ITERATIONS = 60
CHECKPOINT = f"trivariate_matern_components_checkpoint_{JOINT}_xyz.json"
fits = {int(k): v for k, v in json.load(open(CHECKPOINT)).items()} if os.path.exists(CHECKPOINT) else {}
if fits:
    print(f"resuming from {CHECKPOINT}: {len(fits)} gait points done")
torch.cuda.reset_peak_memory_stats()


def fit_gait_point(t, theta0):
    # maximum likelihood estimation of the trivariate model for the outputs at gait point t
    model.set_outputs(Z_train[:, :, t])
    best = {"nll": np.inf, "theta": None, "evaluations": 0}

    def objective(theta):
        # a trial point far from the best point so far is rejected without a factorization
        if best["theta"] is not None and np.max(np.abs(theta - best["theta"])) > 0.7:
            return best["nll"] + 1e6 * float(np.max(np.abs(theta - best["theta"])))
        nll, sigma2, count, stored_gb, precision = negative_log_likelihood(theta, model)
        best["evaluations"] += 1
        if nll < best["nll"]:
            best.update(nll=nll, theta=theta.copy(), sigma2=sigma2, count=count, stored_gb=stored_gb, precision=precision)
        return nll

    result = minimize(objective, theta0, method="L-BFGS-B", bounds=bounds, options={"maxiter": MAX_ITERATIONS, "eps": 1e-3})
    best["iterations"], best["message"] = int(result.nit), str(result.message)
    return best


t_start = time.time()
theta_prev = theta_first
for t in GAIT_POINTS:
    if t in fits:
        theta_prev = np.array(fits[t]["theta"])
        continue
    t0 = time.time()
    best = fit_gait_point(t, theta_prev)
    if best["iterations"] == 0 and t != GAIT_POINTS[0]:
        # the line search failed at the warm start; refit from the starting values of the first gait point and keep the better fit
        cold = fit_gait_point(t, theta_first)
        cold["evaluations"] += best["evaluations"]
        if cold["nll"] < best["nll"]:
            best = cold
    model.set_parameters(best["theta"])
    p = model.parameters
    fits[t] = {"theta": best["theta"].tolist(), "nll": best["nll"], "sigma2": best["sigma2"], "count": best["count"],
               "stored_gb": best["stored_gb"], "iterations": best["iterations"], "evaluations": best["evaluations"], "message": best["message"]}
    json.dump({str(k): v for k, v in fits.items()}, open(CHECKPOINT, "w"))
    theta_prev = best["theta"]
    print(f"gait point {t:3d}  negative log likelihood {best['nll']:10.2f}  {best['iterations']:3d} iterations  {best['evaluations']:4d} evaluations  "
          f"blocks fp8/bf16/fp32/fp64 {'/'.join(str(c) for c in best['count'].values())}  "
          f"input ranges {np.round(p['l'], 3).tolist()}  variances [BW²] {np.round(best['sigma2'] * p['v'] * scale_t[:, t]**2, 4).tolist()}  "
          f"nugget effects [BW²] {[f'{x:.1e}' for x in best['sigma2'] * p['eta'] * scale_t[:, t]**2]}  "
          f"cross-correlations fx-fy {p['rho'][0, 1]:.3f}, fx-fz {p['rho'][0, 2]:.3f}, fy-fz {p['rho'][1, 2]:.3f}  [{time.time() - t0:.0f} s]", flush=True)
print(f"{len(GAIT_POINTS)} gait points fitted, {(time.time() - t_start) / 60:.1f} min")

### Estimates over the gait cycle

The estimates of every gait point are collected in one table, in body weights. The table is shown at the gait point of the peak resultant of the median training stride, and the figure follows the input ranges, the variances, the nugget effects and the cross-correlations $\rho_{jk}$ along the gait cycle.

In [ ]:
rows = []
for t in GAIT_POINTS:
    model.set_parameters(np.array(fits[t]["theta"]))
    p, sigma2 = model.parameters, fits[t]["sigma2"]
    row = {"gait_point": t, "negative_log_likelihood": fits[t]["nll"], "iterations": fits[t]["iterations"], "evaluations": fits[t]["evaluations"]}
    for j, c in enumerate(COMPONENTS):
        row[f"input_range_{c}"] = p["l"][j]
        row[f"variance_{c}_BW2"] = sigma2 * p["v"][j] * scale_t[j, t]**2
        row[f"nugget_{c}_BW2"] = sigma2 * p["eta"][j] * scale_t[j, t]**2
    row.update({"rho_fx_fy": p["rho"][0, 1], "rho_fx_fz": p["rho"][0, 2], "rho_fy_fz": p["rho"][1, 2]})
    row.update({f"blocks_{name}": fits[t]["count"][name] for name in FORMAT_NAMES})
    rows.append(row)
estimates = pd.DataFrame(rows).set_index("gait_point")
estimates.to_csv(f"trivariate_matern_estimates_by_gait_point_{JOINT}_xyz.csv")

t_peak = int(np.median(curve(y_train, f"{JOINT}_jcf_BW").argmax(axis=1)))              # gait point of the peak resultant of the median stride
t_show = min(GAIT_POINTS, key=lambda t: abs(t - t_peak))
e = estimates.loc[t_show]
print(f"estimates at gait point {t_show}, the peak of the resultant   (negative log likelihood {e['negative_log_likelihood']:,.1f})")
shown = pd.DataFrame({c: {"input range [standardized units]": f"{e[f'input_range_{c}']:.3f}", "variance [BW²]": f"{e[f'variance_{c}_BW2']:.4f}",
                          "nugget effect [BW²]": f"{e[f'nugget_{c}_BW2']:.2e}"} for c in COMPONENTS})
shown.columns.name = f"{JOINT} JCF component"
shown

In [ ]:
pd.DataFrame([[1.0, e["rho_fx_fy"], e["rho_fx_fz"]], [e["rho_fx_fy"], 1.0, e["rho_fy_fz"]], [e["rho_fx_fz"], e["rho_fy_fz"], 1.0]],
             index=COMPONENTS, columns=COMPONENTS).round(3)

In [ ]:
COMPONENT_COLOURS = {"fx": "#0072B2", "fy": "#D55E00", "fz": "#009E73"}
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for c in COMPONENTS:
    axes[0][0].plot(estimates.index, estimates[f"input_range_{c}"], color=COMPONENT_COLOURS[c], lw=2, label=c)
    axes[0][1].plot(estimates.index, estimates[f"variance_{c}_BW2"], color=COMPONENT_COLOURS[c], lw=2, label=c)
    axes[1][0].plot(estimates.index, estimates[f"nugget_{c}_BW2"], color=COMPONENT_COLOURS[c], lw=2, label=c)
for pair, colour in zip(["rho_fx_fy", "rho_fx_fz", "rho_fy_fz"], ["#000000", "#CC79A7", "#56B4E9"]):
    axes[1][1].plot(estimates.index, estimates[pair], color=colour, lw=2, label=pair.replace("rho_", "").replace("_", "-"))
axes[1][1].axhline(0, color="grey", lw=0.8)
for ax, title in zip(axes.flat, ["input range [standardized units]", "variance [BW²]", "nugget effect [BW²]", "cross-correlation"]):
    ax.set_title(title, loc="left"); ax.axvline(62, color="grey", lw=0.8, ls=":"); ax.grid(alpha=0.3); ax.legend(frameon=False, fontsize=9)
axes[1][0].set_yscale("log")
for ax in axes[1]:
    ax.set_xlabel("Gait cycle [%]")
fig.suptitle(f"Trivariate Matérn estimates for the {JOINT} JCF components along the gait cycle", fontsize=12)
fig.tight_layout()

### Precision assigned to each block

The figure shows the storage precision that the rule assigned to every block of the Cholesky factor at the estimates of the gait point of the peak resultant, in the layout of the per-block precision figure of Salvaña (2026). The diagonal blocks are always kept in `fp64`, and the percentages in the title refer to the off-diagonal blocks. The summary below the figure counts the formats over the fits of all gait points.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

PRECISION_COLORS = {"fp8": "#5E3C99", "bf16": "#1FBEC0", "fp32": "#F26F22", "fp64": "#4D4D4D"}     # purple / teal / orange / dark gray

def plot_precision(precision, count, N, t, block_size=120):
    nB, total = precision.shape[0], sum(count.values())
    cmap = ListedColormap([PRECISION_COLORS[name] for name in FORMAT_NAMES])
    cmap.set_bad("white")
    fig, ax = plt.subplots(figsize=(6.4, 6.6), constrained_layout=True)
    ax.imshow(np.ma.masked_invalid(precision), cmap=cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N), origin="upper", interpolation="nearest")
    ax.set_title("   ".join(f"{name} {100 * count[name] / total:.1f}%" for name in FORMAT_NAMES), pad=12)
    ax.set_xlabel("block column $k$"); ax.set_ylabel("block row $i$")
    fig.legend(handles=[Patch(facecolor=PRECISION_COLORS[name], label=name) for name in FORMAT_NAMES], loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f"Per-block precision of the factor at gait point {t}\n(block size $n_b$ = {block_size}, {nB} $\\times$ {nB} blocks, $N$ = {N:,})", fontsize=12)
    fig.savefig(f"trivariate_precision_per_block_{JOINT}_xyz_t{t:03d}.png", dpi=200, bbox_inches="tight")
    plt.show()

model.set_outputs(Z_train[:, :, t_show])
model.set_parameters(np.array(fits[t_show]["theta"]))
_, _, count_show, precision_show = mixed_precision_cholesky(model)
plot_precision(precision_show, count_show, model.N, t_show)

total_blocks = estimates[[f"blocks_{name}" for name in FORMAT_NAMES]].sum()
pd.Series({"N per gait point": f"{model.N:,}", "gait points fitted": len(GAIT_POINTS), "blocks": f"{len(model.blocks)} x {len(model.blocks)}",
           "off-diagonal blocks in fp8 / bf16 / fp32 / fp64, all fits": " / ".join(str(int(v)) for v in total_blocks),
           "stored Cholesky factor per fit [MB]": f"{np.mean([fits[t]['stored_gb'] for t in GAIT_POINTS]) * 1e3:.1f}",
           "Cholesky factor in fp64 [MB]": f"{model.N * (model.N + 1) / 2 * 8 / 1e6:.1f}",
           "covariance matrix in fp64 [MB]": f"{model.N * model.N * 8 / 1e6:.1f}",
           "peak GPU memory [MB]": f"{torch.cuda.max_memory_allocated() / 1e6:.1f}"}, name="mixed-precision factorization").to_frame()

## Prediction

For each test stride the emulator receives its feature vector $\mathbf{x}_*$ only, and the three components are predicted jointly at every gait point with the model fitted at that gait point. Let $\mathbf{k}_*$ be the $N \times 3$ matrix of covariances between the three components at $\mathbf{x}_*$ and the $N$ training points, given by the covariance function at the estimates without the nugget effect, and let $\bar{\mathbf{K}}_{**}$ be the $3 \times 3$ matrix of the cross-correlations $\rho_{jk}$ scaled by the variance ratios. The predictive distribution of the standardized residuals of the stride at gait point $t$ is Gaussian with mean and covariance
$$
\mathbb{E}[\mathbf{z}_*] = \mathbf{k}_*^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y}, \qquad \operatorname{Cov}[\mathbf{z}_*] = \hat{\sigma}_1^2 \big( \bar{\mathbf{K}}_{**} - \mathbf{k}_*^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{k}_* + \operatorname{diag}(\eta_j) \big),
$$
and both linear solves use the mixed-precision Cholesky factor of that gait point. The prediction of component $j$ in body weights is $m_j(t) + s_j(t)\, \mathbb{E}[z_{*j}]$, its standard deviation is $s_j(t)$ times the square root of the $j$-th diagonal entry of the covariance, and the 95 % prediction interval is the mean plus and minus $1.96$ standard deviations. Collected over the gait points, the predictions form the three component curves of the stride. The OpenSim components of the test strides are read only in the next section, to score the predictions.

In [ ]:
def predict(model, L, F_new, sigma2):
    # predictive mean and standard deviation of the three standardized components of every new stride
    Fn = torch.tensor(F_new, dtype=torch.float64, device=DEVICE)
    Ft = torch.tensor(F_train, dtype=torch.float64, device=DEVICE)
    h = torch.cdist(Fn, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F_train.shape[1]**0.5    # (new strides, training strides)
    n_new = Fn.shape[0]
    rows = torch.arange(n_new * 3, device=DEVICE)
    new_component, new_stride = rows // n_new, rows % n_new
    pair = (new_component[:, None], model.component[None, :])
    K_star = model.amplitude[pair] * matern(h[new_stride][:, model.stride] * model.inverse_l[pair])     # (3 n_new, N)
    alpha = solve(L, model.blocks, model.y).reshape(-1)
    mean = (K_star @ alpha).reshape(3, n_new)
    V = solve(L, model.blocks, K_star.T)                                                                # Sigma^{-1} k_*, (N, 3 n_new)
    explained = (K_star * V.T).sum(dim=1).reshape(3, n_new)
    prior = torch.diagonal(model.amplitude)[:, None] + torch.tensor(model.parameters["eta"], device=DEVICE)[:, None]
    var = sigma2 * (prior - explained)
    return mean.cpu().numpy(), np.sqrt(np.maximum(var.cpu().numpy(), 1e-12))


pred_mean = np.full(Y_test.shape, np.nan)                                               # (3 components, test strides, 101 gait points), in BW
pred_sd = np.full(Y_test.shape, np.nan)
for t in GAIT_POINTS:
    model.set_outputs(Z_train[:, :, t])
    model.set_parameters(np.array(fits[t]["theta"]))
    L_fit, _, _, _ = mixed_precision_cholesky(model)
    z_mean, z_sd = predict(model, L_fit, F_test, fits[t]["sigma2"])
    pred_mean[:, :, t] = mean_t[:, t][:, None] + scale_t[:, t][:, None] * z_mean
    pred_sd[:, :, t] = scale_t[:, t][:, None] * z_sd
pred_mean.shape

### Results

The out-of-sample scores on the 140 strides of the three held-out participants, per component and over all fitted gait points: the root mean squared error, the continuous ranked probability score of the Gaussian predictive distribution, and the coverage of the 95 % interval, all in body weights except the coverage.

In [ ]:
from scipy.stats import norm

def crps(y, mu, sd):
    z = (y - mu) / sd
    return np.mean(sd * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi)))

G = GAIT_POINTS
inside = np.abs(Y_test[:, :, G] - pred_mean[:, :, G]) <= 1.96 * pred_sd[:, :, G]
results = pd.DataFrame({"RMSE [BW]": np.sqrt(np.mean((pred_mean[:, :, G] - Y_test[:, :, G])**2, axis=(1, 2))),
                        "CRPS [BW]": [crps(Y_test[j][:, G], pred_mean[j][:, G], pred_sd[j][:, G]) for j in range(3)],
                        "95% coverage [%]": inside.mean(axis=(1, 2)) * 100}, index=COMPONENTS).T
results.columns.name = f"{JOINT} JCF component"
results.round(3)

### Predicted curves

One stride per held-out participant, the one with the median error, for each component: the OpenSim curve, the emulator's prediction and its 95 % interval.

In [ ]:
subjects_test = sorted(x_test["subject"].unique())
labels = {"fx": "$f_x$ anterior-posterior", "fy": "$f_y$ axial", "fz": "$f_z$ medio-lateral"}
fig, axes = plt.subplots(3, len(subjects_test), figsize=(6 * len(subjects_test), 11), sharex=True)
for j, c in enumerate(COMPONENTS):
    y, mu, sd = Y_test[j][:, G], pred_mean[j][:, G], pred_sd[j][:, G]
    for col, s in enumerate(subjects_test):
        rows = np.where(x_test["subject"].to_numpy() == s)[0]
        err = np.sqrt(np.mean((mu[rows] - y[rows])**2, axis=1))
        i = rows[np.argsort(err)[len(err) // 2]]
        ax = axes[j][col]
        ax.fill_between(G, mu[i] - 1.96 * sd[i], mu[i] + 1.96 * sd[i], color="#f3c9b0", lw=0, label="95% interval")
        ax.plot(G, y[i], color="#1f3b5c", lw=2.2, label="OpenSim (Truth)")
        ax.plot(G, mu[i], color="#D55E00", lw=2.2, ls="--", label="emulator")
        ax.set_title(f"{c}: {s.replace('NIH_ACL_', '')}, {x_test.loc[i, 'block']}, stride {x_test.loc[i, 'stride_in_block']}, {x_test.loc[i, 'belt_speed_mps']:g} m/s", loc="left", fontsize=10)
        ax.text(0.98, 0.97, f"RMSE {err[len(err) // 2]:.2f} BW, {np.mean(np.abs(y[i] - mu[i]) <= 1.96 * sd[i]) * 100:.0f}% inside", transform=ax.transAxes, ha="right", va="top", fontsize=9)
        ax.grid(alpha=0.3)
        if col == 0:
            ax.set_ylabel(f"{labels[c]} [BW]")
        if j == 2:
            ax.set_xlabel("Gait cycle [%]")
axes[0][0].legend(frameon=False, fontsize=9, loc="lower left")
fig.tight_layout()
fig.savefig(f"trivariate_predicted_curves_{JOINT}_xyz.png", dpi=200, bbox_inches="tight")

## References

Apanasovich, T. V., Genton, M. G. and Sun, Y. (2012). A valid Matérn class of cross-covariance functions for multivariate random fields with any number of components. *Journal of the American Statistical Association*, 107, 180-193.

Gneiting, T., Kleiber, W. and Schlather, M. (2010). Matérn cross-covariance functions for multivariate random fields. *Journal of the American Statistical Association*, 105, 1167-1177.

Salvaña, M. L. O. (2026). Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes. In *Proceedings of the SC26 Workshops, HPC for Environmental and Earth Sciences (HPC4EES)*, Chicago, IL, USA, November 15-20, 2026.